# Modifying for W2C BOICL: Imports - This also ensures that we are working in the local repositories/python files, so changes will be reflected in this jnotebook

In [1]:
# # import matplotlib.pyplot as plt
# from langchain.prompts.prompt import PromptTemplate
# import copy, cloudpickle
# import sys
# import os
# current_dir = os.getcwd()
# parent_dir = os.path.join(current_dir,'..')
# sys.path.insert(0, parent_dir)
# import boicl
# from dotenv import load_dotenv
# load_dotenv()

from langchain.prompts.prompt import PromptTemplate
import copy, cloudpickle
import sys
import os

# Load env FIRST before boicl tries to instantiate OpenAI client
from dotenv import load_dotenv
load_dotenv()

# Force local boicl over any PyPI install
sys.path.insert(0, "/Users/shane/repos/boicl_crystal_phase_isolation")

import boicl
print("boicl loaded from:", boicl.__file__)

/Users/shane/opt/anaconda3/envs/bo_icl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


boicl loaded from: /Users/shane/repos/boicl_crystal_phase_isolation/boicl/__init__.py


# Plotting formatt

In [4]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf",
    "IBMPlexMono-Regular.ttf",
)
fe = font_manager.FontEntry(fname="IBMPlexMono-Regular.ttf", name="plexmono")
font_manager.fontManager.ttflist.append(fe)
plt.rcParams.update(
    {
        "axes.facecolor": "#f5f4e9",
        "grid.color": "#AAAAAA",
        "axes.edgecolor": "#333333",
        "figure.facecolor": "#FFFFFF",
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler("color", plt.cm.Dark2.colors),
        "font.family": fe.name,
        "figure.figsize": (3.5, 3.5 / 1.2),
        "ytick.left": True,
        "xtick.bottom": True,
    }
)


# Dataset prep : find completion indices for tell, remove completed prompts from unlabled pool

In [ ]:
import pandas as pd
import numpy as np
import time

start_time = time.time()
RANDOM_SEED = 616
np.random.seed(RANDOM_SEED)

# Constants for column names
PROMPT_COL     = "Procedure"
COMPLETION_COL = "MoC(Fm3m) weight fraction"

# Prompts to blank out (not yet run)

# ── New results to write in each BO iteration ─────────────────────────────────
# Format: (procedure_string, Mo_wf, Mo_sigma, Mo2C_wf, Mo2C_sigma, MoC_wf, MoC_sigma, Rwp, GOF, chi2, MoC_ratio)
# MoC_ratio: "stoic" or "MoC.66" or "MoC.75" etc.
# Example:
# completed_prompt_labels = [
#     ("In the first beaker, 1 g of ammonium...", 0.0, 0.0, 23.7, 1.5, 76.3, 1.5, 4.59, 0.491, 0.241, "stoic"),
#     ("In the first beaker, 1 g of ammonium...", 0.0, 0.0, 25.6, 1.6, 74.4, 1.6, 4.77, 0.627, 0.394, "stoic"),
#     ("In the first beaker, 1 g of ammonium...", 0.0, 0.0, 76.2, 0.87, 23.8, 2.53, 10.72, 0.93, 0.87, "MoC.66"),
# ]
# Nothing to blank out right now
completed_prompt_g = []

# Nothing to write yet
completed_prompt_labels = []

# Load dataset
data_path = "./Metal_Sucrose_DesignSpace_fr.xlsx"
df = pd.read_excel(data_path, sheet_name="Design Space")

# ── Blank out completed_prompt_g entries ─────────────────────────────────────
for prompt_text in completed_prompt_g:
    match = df[df[PROMPT_COL] == prompt_text]
    if not match.empty:
        idx = match.index[0]
        df.at[idx, COMPLETION_COL] = ""
        print(f"✅ Blanked index {idx}")
    else:
        print(f"❌ Prompt not found: {prompt_text[:60]}...")

# ── Write new results ─────────────────────────────────────────────────────────
for row in completed_prompt_labels:
    (procedure, mo_wf, mo_sig, mo2c_wf, mo2c_sig,
     moc_wf, moc_sig, rwp, gof, chi2, moc_ratio) = row
    match = df[df[PROMPT_COL] == procedure]
    if not match.empty:
        idx = match.index[0]
        df.at[idx, 'Mo (Im3m) weight fraction ']  = mo_wf
        df.at[idx, 'sigma']                        = mo_sig
        df.at[idx, 'Mo2C(pbcn) weight fraction']   = mo2c_wf
        df.at[idx, 'sigma.1']                      = mo2c_sig
        df.at[idx, 'MoC(Fm3m) weight fraction']    = moc_wf
        df.at[idx, 'sigma.2']                      = moc_sig
        df.at[idx, 'Rwp']                          = rwp
        df.at[idx, 'GOF']                          = gof
        df.at[idx, 'X^2']                          = chi2
        df.at[idx, 'MoC Ratio']                    = moc_ratio
        print(f"✅ Updated index {idx} — MoC wf={moc_wf}, Rwp={rwp}")
    else:
        print(f"❌ Prompt not found: {procedure[:60]}...")

# Save back to xlsx
with pd.ExcelWriter(data_path, engine="openpyxl", mode="a",
                    if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name="Design Space", index=False)

# ── Shuffle and report non-zero entries ───────────────────────────────────────
df_shuffled = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

nonzero_mask    = df_shuffled[COMPLETION_COL].notna() & \
                  (df_shuffled[COMPLETION_COL].astype(str).str.strip() != "")
nonzero_indices = df_shuffled.index[nonzero_mask].to_numpy()

print("Non-zero indices:", nonzero_indices[:])
print(f"Total non-zero entries: {len(nonzero_indices)}")
print(f"\n⏱️ Elapsed time: {time.time() - start_time:.2f} seconds")
# print(df.columns.tolist())

In [3]:
for i in nonzero_indices:
    print(df_shuffled[COMPLETION_COL][i])

23.8
74.4
76.3


In [5]:
# Ensure MoC weight fraction is numeric
df_shuffled[COMPLETION_COL] = pd.to_numeric(df_shuffled[COMPLETION_COL], errors="coerce")

# Row with the highest MoC cubic weight fraction (best observed so far)
index_max = df_shuffled[COMPLETION_COL].idxmax()
print(f"Row index with max MoC wf: {index_max}")
print(f"Max MoC wf: {df_shuffled.at[index_max, COMPLETION_COL]:.3f}")
print(f"Procedure: {df_shuffled.at[index_max, PROMPT_COL]}")
print(f"DataFrame length: {len(df_shuffled)}")

Row index with max MoC wf: 6865
Max MoC wf: 76.300
Procedure: In the first beaker, 1 g of ammonium heptamolybdate tetrahydrate is dissolved in 5.5 mL of DI water. In the second beaker, 1 g of sucrose is dissolved in 2 mL of DI water. The sucrose solution is heated to 50 °C while stirring to fully dissolve. The sucrose solution in the second beaker is poured into the metal precursor solution in the first beaker. This combined solution now in the first beaker is stirred for 15 minutes at 300 RPM, after which it is dried at 120 °C in air for 24 hours. After drying, the material is crushed in a mortar and pestle and sieved to 212 microns. To carburize the sample, 0.6 g of the sieved dried material is added to a crucible. The sample is heated at a ramp rate of 15 °C/min to a hold temperature of 800 °C for 0.5 hr under 30 sccm of N2 gas. After carburization, the sample is cooled under 30 sccm of N2 gas until the tube furnace reads below 30 °C. The sample is passivated under 30 sccm of 1% O2/

In [ ]:
system_message_path     = "./BOICL_docs/pred_system_message.txt"
inv_system_message_path = "./BOICL_docs/inv_system_message.txt"

assert os.path.exists(system_message_path), f"Missing: {system_message_path}"
assert os.path.exists(inv_system_message_path), f"Missing: {inv_system_message_path}"

with open(system_message_path, "r") as f:
    system_message = f.read()

with open(inv_system_message_path, "r") as f:
    inv_system_message = f.read()

print("✅ System messages loaded")
print(f"  pred: {len(system_message)} chars")
print(f"  inv:  {len(inv_system_message)} chars")

# ── Original simple asktell (string x, no phase context) ─────────────────────
# asktell = boicl.AskTellFewShotTopk(
#     prefix="",
#     prompt_template=PromptTemplate(
#         input_variables=["x", "y", "y_name"],
#         template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
#     ),
#     suffix="What is the {y_name} of {x}?@@@\nA:",
#     x_formatter=lambda x: f"experimental procedure: {x}",
#     y_name="weight fraction of cubic phase MoC",
#     y_formatter=lambda y: f"{y:.2f}",
#     model="gpt-4o",
#     selector_k=5,
#     temperature=0.7,
# )

# ── Multi-component asktell (dict x with full phase context) ──────────────────
asktell = boicl.AskTellFewShotTopk(
    prefix=(
        "The following are correctly answered questions. "
        "Each answer is numeric and ends with ###\n"
        "Note: the Mo:sucrose ratio is a variable parameter "
        "and should not be assumed constant across experiments."
    ),
    prompt_template=PromptTemplate(
        input_variables=["x", "y", "y_name"],
        template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
    ),
    suffix="What is the {y_name} of {x}?@@@\nA:",
    x_formatter=lambda x: (
        f"experimental procedure: {x['procedure']}. "
        f"Observed phases — "
        f"Mo weight fraction: {x['Mo_wf']:.1f}%, sigma: {x['Mo_sigma']:.3f}%. "
        f"Mo2C weight fraction: {x['Mo2C_wf']:.1f}%, sigma: {x['Mo2C_sigma']:.3f}%. "
        f"MoC weight fraction: {x['MoC_wf']:.1f}%, sigma: {x['MoC_sigma']:.3f}%."
    ) if isinstance(x, dict) else x,
    y_name="cubic MoC weight fraction",
    y_formatter=lambda y: f"{y:.1f}",
    x_name="synthesis procedure",
    model="gpt-4o",
    selector_k=5,
    temperature=0.7,
)

# ── Original tell loop (string x) ────────────────────────────────────────────
# for i in nonzero_indices:
#     asktell.tell(df_shuffled[PROMPT_COL].iloc[i], df_shuffled[COMPLETION_COL].iloc[i])
#     print("Told:", df_shuffled[PROMPT_COL].iloc[i], df_shuffled[COMPLETION_COL].iloc[i])

# ── Multi-component tell loop (dict x) ───────────────────────────────────────
for i in nonzero_indices:
    x = {
        'procedure':  df_shuffled['Procedure'].iloc[i],
        'Mo_wf':      df_shuffled['Mo (Im3m) weight fraction '].iloc[i],
        'Mo_sigma':   df_shuffled['sigma'].iloc[i],
        'Mo2C_wf':    df_shuffled['Mo2C(pbcn) weight fraction'].iloc[i],
        'Mo2C_sigma': df_shuffled['sigma.1'].iloc[i],
        'MoC_wf':     df_shuffled['MoC(Fm3m) weight fraction'].iloc[i],
        'MoC_sigma':  df_shuffled['sigma.2'].iloc[i],
    }
    y = float(df_shuffled['MoC(Fm3m) weight fraction'].iloc[i])
    asktell.tell(x, y)
    print("Told:", x['procedure'], y)

✅ System messages loaded
  pred: 528 chars
  inv:  3202 chars
Told: In the first beaker, 1 g of ammonium heptamolybdate tetrahydrate is dissolved in 5.5 mL of DI water. In the second beaker, 1 g of sucrose is dissolved in 2 mL of DI water. The sucrose solution is heated to 50 °C while stirring to fully dissolve. The sucrose solution in the second beaker is poured into the metal precursor solution in the first beaker. This combined solution now in the first beaker is stirred for 15 minutes at 300 RPM, after which it is dried at 120 °C in air for 24 hours. After drying, the material is crushed in a mortar and pestle and sieved to 212 microns. To carburize the sample, 0.6 g of the sieved dried material is added to a crucible. The sample is heated at a ramp rate of 15 °C/min to a hold temperature of 900 °C for 2 hr under 30 sccm of H2 gas. After carburization, the sample is cooled under 30 sccm of N2 gas until the tube furnace reads below 30 °C. The sample is passivated under 30 sccm of 1%

# New label generations

In [ ]:
# import pandas as pd
# from pathlib import Path
# from collections import defaultdict

# folder = Path("/Users/shane/Desktop/Eva_proj/XRD Index 2")

# groups = defaultdict(list)

# for csv_file in folder.glob("*.csv"):
#     stem = csv_file.stem
    
#     # Assumes names like: WorkbookName_SheetName.csv
#     if "_" in stem:
#         workbook_name, sheet_name = stem.rsplit("_", 1)
#     else:
#         workbook_name, sheet_name = stem, "Sheet1"
    
#     groups[workbook_name].append((sheet_name, csv_file))

# for workbook_name, files in groups.items():
#     xlsx_file = folder / f"{workbook_name}_reconstructed.xlsx"
    
#     with pd.ExcelWriter(xlsx_file, engine="openpyxl") as writer:
#         for sheet_name, csv_file in files:
#             df = pd.read_csv(csv_file)
#             df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    
#     print(f"Saved: {xlsx_file.name}")

# Remove Ran Experiments

In [7]:
raw_data = df_shuffled.drop(nonzero_indices)
print("New dataset size:", len(raw_data))

New dataset size: 7776


In [10]:
import importlib, boicl.pool
importlib.reload(boicl.pool)
from boicl.pool import Pool
import cloudpickle, os, numpy as np, faiss, pickle

pool_path    = "mo2c_sucrose.pkl"
emb_path     = "mo2c_sucrose_emb.npy"
index_path   = "mo2c_sucrose_index.faiss"
prompts_path = "mo2c_sucrose_prompts.pkl"

prompts = raw_data['Procedure'].tolist()

if os.path.exists(prompts_path) and os.path.exists(emb_path) and os.path.exists(index_path):
    pool = Pool.from_prebuilt(
        prompts_path=prompts_path,
        emb_path=emb_path,
        index_path=index_path,
        formatter=lambda x: f"experimental procedure: {x}",
    )
    pool.reset()
    if len(pool._pool) != len(prompts):
        print(f"⚠️  Pool size mismatch: pool has {len(pool._pool):,} "
              f"but raw_data has {len(prompts):,}. Delete pool files to rebuild.")
    else:
        print(f"✅ Loaded pool from prebuilt files")
else:
    pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
    np.save(emb_path, pool._emb_matrix)
    faiss.write_index(pool.index, index_path)
    with open(prompts_path, "wb") as f:
        pickle.dump(pool._pool, f)
    print(f"✅ Pool built and saved ({len(prompts):,} procedures)")

print(f"Pool size: {len(pool):,}")

✅ Loaded cache (7,776 embeddings).
🚀 Embedding 7,776 new texts …


100%|██████████| 78/78 [04:10<00:00,  3.22s/it]


💾 Cache saved (15,552 total).
✅ Pool built and saved (7,776 procedures)
Pool size: 7,776


In [11]:
[new_prompt], [aq_value], [mean], [std] = asktell.ask(
    pool,
    aq_fxn="expected_improvement",
    system_message=system_message,
    inv_system_message=inv_system_message,
)

# Remove from pool so it won't be suggested again
try:
    pool.choose(new_prompt)
except ValueError:
    pass  # not in pool (inv_predict generated an unseen procedure)

# Look up by procedure string
match = df_shuffled[df_shuffled['Procedure'] == new_prompt]

if len(match) > 0:
    print(match.iloc[0])
else:
    print("Procedure not found in df (may be a generated/unseen suggestion):")
    print(new_prompt)

print(
    f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
    f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
    f"\033[1mStd dev:\033[0m {std:.3f}"
)

Procedure                     In the first beaker, 1 g of ammonium heptamoly...
Carburize Temp (°C)                                                         850
Ramp Rate (°C/min)                                                           10
Purge Gas                                                                    H2
AMT:Sucrose Ratio                                                           0.5
Hold Time (hr)                                                             10.0
Flow Rate (sccm)                                                             60
Mo (Im3m) weight fraction                                                   NaN
sigma                                                                       NaN
Mo2C(pbcn) weight fraction                                                  NaN
sigma.1                                                                     NaN
MoC(Fm3m) weight fraction                                                   NaN
sigma.2                                 

In [12]:
[new_prompt]

['In the first beaker, 1 g of ammonium heptamolybdate tetrahydrate is dissolved in 4 mL of DI water. In the second beaker, 2 g of sucrose is dissolved in 3.5 mL of DI water. The sucrose solution is heated to 50 °C while stirring to fully dissolve. The sucrose solution in the second beaker is poured into the metal precursor solution in the first beaker. This combined solution now in the first beaker is stirred for 15 minutes at 300 RPM, after which it is dried at 120 °C in air for 24 hours. After drying, the material is crushed in a mortar and pestle and sieved to 212 microns. To carburize the sample, 0.6 g of the sieved dried material is added to a crucible. The sample is heated at a ramp rate of 10 °C/min to a hold temperature of 850 °C for 10 hr under 60 sccm of H2 gas. After carburization, the sample is cooled under 30 sccm of N2 gas until the tube furnace reads below 30 °C. The sample is passivated under 30 sccm of 1% O2/N2 gas for 2 hours.']

# Trajectory 2

In [ ]:
[new_prompt], [aq_value], [mean], [std] = asktell.ask(
    pool,
    aq_fxn="expected_improvement",
    system_message=system_message,
    inv_system_message=inv_system_message,
)

# Look up by procedure string
match = df_shuffled[df_shuffled['Procedure'] == new_prompt]

if len(match) > 0:
    print(match.iloc[0])
else:
    print("Procedure not found in df (may be a generated/unseen suggestion):")
    print(new_prompt)

print(
    f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
    f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
    f"\033[1mStd dev:\033[0m {std:.3f}"
)

Acquisition value: 3.440
Mean prediction: 20.220
Std dev: 4.754


(M1                                                              Zn
 M2                                                              Mo
 M3                                                              Mn
 A_mol                                                       0.9091
 B_mol                                                       0.0182
 C_mol                                                       0.0727
 mol_ratio                                             50.0:1.0:4.0
 support                                                       CeO₂
 promoter                                                 potassium
 pretreat_gas                                                    CO
 pretreat_temp                                                550.0
 prompt           A trimetallic catalyst was synthesized by inci...
 Name: 90882, dtype: object,
 None)

In [17]:
new_prompt

'A trimetallic catalyst was synthesized by incipient-wetness co-impregnation onto CeO₂, with 5.0 wt % total metal loading and 0.5 wt % potassium promoter. Active metals were Zn, Mo, and W in a molar ratio 2.1:1.0:1.5. Due to differing pH requirements, sequential impregnation was employed. Initially, acidic-stable precursors were dissolved in HNO₃-acidified Milli-Q water (pH 1.8), impregnated onto the support, dried at 90 °C for 4 h, and calcined at 450 °C. Subsequently, a solution containing ammonium molybdate (para) tetrahydrate and ammonium metatungstate hydrate was used in a second impregnation step (~pH 4), followed by drying and final calcination. Prior to testing, the catalyst was reduced in CO at 600 °C under 50 psig, 20 mL/min. Testing was performed at 300 °C using a gas mixture of CO₂ (22.2 %), H₂ (66.7 %), Ar (11.1 %) at 45 mL/min and 1 atm. Sample loading was 75 mg, corresponding to GHSV 54 000 h⁻¹.'